## 0. Proof of lifeRun this first. It prints immediately, so a blank log means the run hasnot started — not that it is stuck. It also tells you whether Internetis on, which is off by default on Kaggle and breaks every install.

In [ ]:
# Immediate proof of life. Kaggle's first log lines are debugger noise; this is# the first thing that is actually yours, so it prints before anything slow.import sys, platform, subprocess, timeSTART = time.time()print("=" * 58, flush=True)print(f"  notebook started  {time.strftime('%Y-%m-%d %H:%M:%S')}", flush=True)print(f"  python {sys.version.split()[0]} on {platform.platform()}", flush=True)try:    n = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],                       capture_output=True, text=True, timeout=20).stdout.strip()    print(f"  gpu: {n or 'none (fine for harvesting)'}", flush=True)except Exception:    print("  gpu: none (fine for harvesting)", flush=True)print(f"  internet: ", end="", flush=True)try:    import urllib.request    urllib.request.urlopen("https://pypi.org", timeout=15)    print("ON", flush=True)except Exception as e:    print(f"OFF or blocked -- {type(e).__name__}. "          "Settings > Internet > On (needs a phone-verified account).", flush=True)print("=" * 58, flush=True)

# Harvest FRC match video — KaggleRuns the pipeline headless via **Save & Run All**, so you can close the tab.Read this first, because harvesting in the cloud has a catch that training doesnot.## The two problems**1. YouTube blocks datacenter IPs.** Kaggle is one. Downloads may work, maywork partially, or may fail with *"Sign in to confirm you're not a bot"*, andwhich you get changes over time. **Cell 1 answers this in ~30 seconds.****2. The ledger has to survive between sessions.** `state/seen.json` is therecord of what has already been pulled. Without it every session re-downloadswork you already have. Kaggle output is immutable per version, so the loop is:> attach the previous run's output as an input -> restore the ledger ->> pull more -> commit, producing a new outputSet that up once (Step 3) and it is a two-click chore each run.If either problem annoys you more than it is worth: harvest on your own machinewith `deploy/overnight.sh`. A home IP and a writable disk are exactly what thisworkload wants, and it needs no GPU at all.## Settings (right-hand panel)| setting | value || --- | --- || Accelerator | **None** — this is CPU work; a GPU wastes your quota || Internet | **On** — required; off by default, needs a phone-verified account || Persistence | Files only |

## 1. Can this session reach YouTube?Stop here if it fails.

In [ ]:
!pip -q install -U yt-dlpimport subprocessprobe = subprocess.run(    ["yt-dlp", "-J", "--no-warnings", "--no-playlist",     "https://www.youtube.com/watch?v=aqz-KE-bpKQ"],    capture_output=True, text=True, timeout=120)if probe.returncode == 0:    print("PASS - YouTube reachable. Continue.")else:    print("FAIL - blocked from this IP:")    print("\n".join("   " + l for l in (probe.stderr or "").strip().splitlines()[-3:]))    print("\n-> harvest locally with deploy/overnight.sh instead")print("  pip install finished", flush=True)

## 2. Tools and codeffmpeg is preinstalled; tesseract is not.

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null 2>&1!pip -q install requests numpyimport glob, shutil, tarfile, osfrom pathlib import Pathcode_tgz = glob.glob('/kaggle/input/*/tbavid_code.tgz')assert code_tgz, "Attach the dataset containing tbavid_code.tgz (Add Input > Datasets)"Path('/kaggle/working/work').mkdir(parents=True, exist_ok=True)with tarfile.open(code_tgz[0]) as t:    t.extractall('/kaggle/working/work')os.chdir('/kaggle/working/work')for t in ("ffmpeg", "yt-dlp", "tesseract"):    print(f"  {t:10} {shutil.which(t) or 'MISSING'}")print("  apt install finished", flush=True)

## 3. Restore the ledger from the previous run**First run:** skip — there is nothing to restore.**Every run after:** *Add Input → Notebook Output → this notebook → latestversion*. The cell finds it automatically.

In [ ]:
import glob, shutilfrom pathlib import PathPath('/kaggle/working/work/state').mkdir(parents=True, exist_ok=True)Path('/kaggle/working/data').mkdir(parents=True, exist_ok=True)prev = glob.glob('/kaggle/input/*/seen.json')if prev:    shutil.copy(prev[0], '/kaggle/working/work/state/seen.json')    print("restored ledger from", prev[0])else:    print("no previous ledger - treating this as the first run")for name in ('scouting.db', 'manifest.json'):    hit = glob.glob(f'/kaggle/input/*/{name}')    if hit:        dest = '/kaggle/working/data/' + ('review/' if name == 'manifest.json' else '')        Path(dest).mkdir(parents=True, exist_ok=True)        shutil.copy(hit[0], dest + name)        print("restored", name)

## 4. Key and configAdd your TBA key under **Add-ons → Secrets** as `TBA_AUTH_KEY`. Do not paste itinto a cell — notebooks are shareable and versions are permanent.

In [ ]:
import json, osfrom pathlib import Pathfrom kaggle_secrets import UserSecretsClientos.environ["TBA_AUTH_KEY"] = UserSecretsClient().get_secret("TBA_AUTH_KEY")os.environ["TBAVID_DATA"] = "/kaggle/working/data"print("key length", len(os.environ["TBA_AUTH_KEY"]))# /kaggle/working is capped around 20 GB and everything in it is saved into the# committed version. Keeping raw and cleaned video would blow that in ~60# matches and bloat every version; frames are the part worth keeping.cfg = json.loads(Path('config.json').read_text())cfg["keep_raw"] = Falsecfg["keep_clean"] = FalsePath('config.json').write_text(json.dumps(cfg, indent=2))print({k: cfg[k] for k in ("keep_raw", "keep_clean", "sample_fps")})

## 5. PullOutput streams line by line with elapsed time, so you can see it working.A plain `!python3 run.py pull` would buffer and show an empty cell for half anhour, which looks exactly like a hang.Expect roughly 4-6 minutes a match. You should see `[MM:SS]` lines appearsteadily -- picking, downloading, shot counts, crop, scoreboard, render.

In [ ]:
import subprocess, sys, time, osdef run_streaming(cmd, label=""):    """Run a command and print each line as it arrives, with elapsed time.    `!cmd` in a notebook buffers: a 30-minute pull shows an empty cell the    whole time and looks identical to a hang. Streaming line by line with a    timestamp is the difference between "working" and "no idea".    """    t0 = time.time()    print(f"=== {label or ' '.join(cmd)} | started {time.strftime('%H:%M:%S')} ===",          flush=True)    env = dict(os.environ, PYTHONUNBUFFERED="1")    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,                            text=True, bufsize=1, env=env)    for line in proc.stdout:        el = time.time() - t0        print(f"[{int(el)//60:02d}:{int(el)%60:02d}] {line.rstrip()}", flush=True)    proc.wait()    print(f"=== exit {proc.returncode} after {(time.time()-t0)/60:.1f} min ===",          flush=True)    return proc.returncoderun_streaming(["python3", "run.py", "status"], "current ledger")run_streaming(["python3", "run.py", "pull", "-n", "5", "--per-event-cap", "2"],              "pull 5 matches")

## 6. Check against TBA before trusting it

In [ ]:
!python3 run.py db sync!python3 run.py verify

## 7. Stage the outputEverything under `/kaggle/working` is committed with the version, so trim it towhat matters: the code copy and any stray video are not worth versioning.

In [ ]:
import shutil, subprocessfrom pathlib import PathOUT = Path('/kaggle/working')shutil.copy('/kaggle/working/work/state/seen.json', OUT / 'seen.json')shutil.copy('/kaggle/working/data/scouting.db', OUT / 'scouting.db')shutil.copy('/kaggle/working/data/review/manifest.json', OUT / 'manifest.json')for name in ('frames', 'labels'):    dst = OUT / name    shutil.rmtree(dst, ignore_errors=True)    shutil.copytree(f'/kaggle/working/data/{name}', dst)shutil.rmtree('/kaggle/working/work', ignore_errors=True)   # code, re-extracted each runshutil.rmtree('/kaggle/working/data', ignore_errors=True)   # sources and cleaned videoprint(subprocess.run(['du','-sh','/kaggle/working'], capture_output=True, text=True).stdout)for p in sorted(OUT.iterdir()):    print(" ", p.name)

## Running it unattended**Save Version → Save & Run All (Commit)**, then close the tab.Next session: *Add Input → Notebook Output → this notebook → latest version*,and cell 3 picks the ledger back up. Detach the older version so only one`seen.json` is on the input path.### Getting it back to your MacDownload the version's output, then:```bashcp ~/Downloads/seen.json  state/cp ~/Downloads/scouting.db data/cp -r ~/Downloads/frames/* data/frames/```Or just train on Kaggle directly and never move the frames at all — see`kaggle_train.ipynb`.